1. Importe les bibliothèques et tes sept fichiers CSV

In [7]:
import pandas as pd
import json

from google.colab import files
from IPython.display import display

fichiers = files.upload()

Saving artif-departement.csv to artif-departement.csv
Saving Incendies.csv to Incendies.csv
Saving mdf_2024.csv to mdf_2024.csv
Saving mdf_2025.csv to mdf_2025.csv
Saving mdf_2026.csv to mdf_2026.csv


In [9]:
from pathlib import Path

# Définition des chemins par défaut
chemin_mos_defaut = Path("/content/donnees_qualite/mos-occupation-du-sol-2025-and-2021-en-79-postes-de-la-region-ile-de-france.csv")
chemin_nom_defaut = Path("/content/donnees_qualite/nomenclature-du-mode-d-occupation-du-sol-francilien-79-postes.csv")

# Retrouver les noms des deux fichiers longs s'ils ont été téléversés, sinon utiliser les chemins par défaut
try:
    nom_mos = next(n for n in fichiers if n.startswith("mos-occupation"))
except StopIteration:
    nom_mos = str(chemin_mos_defaut) if chemin_mos_defaut.exists() else ""

try:
    nom_nomenclature = next(n for n in fichiers if n.startswith("nomenclature"))
except StopIteration:
    nom_nomenclature = str(chemin_nom_defaut) if chemin_nom_defaut.exists() else ""

if not nom_mos or not nom_nomenclature:
    raise FileNotFoundError("Les fichiers MOS ou Nomenclature sont introuvables. Veuillez les téléverser.")

mos = pd.read_csv(nom_mos, sep=";", dtype=str)
_nom_file = nom_nomenclature
nomenclature = pd.read_csv(_nom_file, sep=";", dtype=str)

# Lecture des autres fichiers (avec repli s'ils sont dans donnees_qualite)
def charger_csv(nom_fichier, sep=",", skiprows=0):
    chemin_local = Path(nom_fichier)
    chemin_dossier = Path("/content/donnees_qualite") / nom_fichier
    if chemin_local.exists():
        return pd.read_csv(chemin_local, sep=sep, skiprows=skiprows, dtype=str)
    elif chemin_dossier.exists():
        return pd.read_csv(chemin_dossier, sep=sep, skiprows=skiprows, dtype=str)
    else:
        raise FileNotFoundError(f"Le fichier {nom_fichier} est introuvable.")

artif = charger_csv("artif-departement.csv")
meteo_2024 = charger_csv("mdf_2024.csv")
meteo_2025 = charger_csv("mdf_2025.csv")
meteo_2026 = charger_csv("mdf_2026.csv", sep=";")
bdiff = charger_csv("Incendies.csv", sep=";", skiprows=3)

bases = {
    "MOS": mos,
    "Nomenclature": nomenclature,
    "Artificialisation": artif,
    "Météo 2024": meteo_2024,
    "Météo 2025": meteo_2025,
    "Météo 2026": meteo_2026,
    "BDIFF": bdiff
}

# Retirer les espaces inutiles et repérer les cellules vides
for nom_base, base in bases.items():
    for colonne in base.columns:
        base[colonne] = base[colonne].str.strip().replace("", pd.NA)

print("Les sept fichiers sont correctement chargés et nettoyés.")

Les sept fichiers sont correctement chargés et nettoyés.


2 profil général et les valeurs manquantes

In [10]:
profils = {}

for nom, base in bases.items():
    print(f"\n--- {nom} ---")
    print("Nombre de lignes :", len(base))
    print("Nombre de colonnes :", len(base.columns))
    print("Doublons de lignes exactes :", base.duplicated().sum())

    profil = pd.DataFrame({
        "Colonne": base.columns,
        "Valeurs manquantes": base.isna().sum().values,
        "Remplissage (%)": (
            base.notna().mean() * 100
        ).round(2).values,
        "Valeurs distinctes": base.nunique().values
    })

    profils[nom] = profil
    display(profil)


--- MOS ---
Nombre de lignes : 42115
Nombre de colonnes : 9
Doublons de lignes exactes : 0


,Colonne,Valeurs manquantes,Remplissage (%),Valeurs distinctes
0,Geo Point,0,100.00,42115
1,Geo Shape,305,99.28,41810
2,mos2021,0,100.00,16
3,Poste 2021,0,100.00,16
4,mos2025,0,100.00,1
5,Poste 2025,0,100.00,1
6,insee,0,100.00,1204
7,st_area_sh,0,100.00,42115
8,st_length_,0,100.00,42115



--- Nomenclature ---
Nombre de lignes : 79
Nombre de colonnes : 3
Doublons de lignes exactes : 0


,Colonne,Valeurs manquantes,Remplissage (%),Valeurs distinctes
0,ID,0,100.0,79
1,Poste,0,100.0,79
2,Définition,0,100.0,79



--- Artificialisation ---
Nombre de lignes : 100
Nombre de colonnes : 11
Doublons de lignes exactes : 0


,Colonne,Valeurs manquantes,Remplissage (%),Valeurs distinctes
0,departement_code,0,100.0,100
1,nom,0,100.0,100
2,region_code,0,100.0,17
3,millesimes_1,0,100.0,5
4,pourcent_artif_1,0,100.0,100
5,surface_artif_1,0,100.0,100
6,millesimes_2,0,100.0,5
7,pourcent_artif_2,0,100.0,100
8,surface_artif_2,0,100.0,100
9,flux_surface_1_2,0,100.0,100


... (4 outputs left out)


--- Météo 2026 ---
Nombre de lignes : 12576
Nombre de colonnes : 5
Doublons de lignes exactes : 0


,Colonne,Valeurs manquantes,Remplissage (%),Valeurs distinctes
0,date,0,100.0,131
1,num_dep,0,100.0,96
2,niveau_j1,0,100.0,4
3,niveau_j2,0,100.0,4
4,nom_dep,0,100.0,96



--- BDIFF ---
Nombre de lignes : 347
Nombre de colonnes : 24
Doublons de lignes exactes : 0


,Colonne,Valeurs manquantes,Remplissage (%),Valeurs distinctes
0,Année,0,100.00,19
1,Numéro,0,100.00,345
2,Département,0,100.00,5
3,Code INSEE,0,100.00,117
4,Nom de la commune,0,100.00,117
5,Date de première alerte,0,100.00,345
6,Surface parcourue (m2),0,100.00,80
7,Surface forêt (m2),0,100.00,66
8,Surface maquis garrigues (m2),347,0.00,0
9,Autres surfaces naturelles hors forêt (m2),0,100.00,25


3 Analyse base par base



3.1 Analyse la qualité du MOS


In [11]:
# Correspondance entre codes MOS et libellés
dictionnaire_mos = nomenclature.set_index("ID")["Poste"]

for annee in ["2021", "2025"]:
    libelle_attendu = mos[f"mos{annee}"].map(dictionnaire_mos)

    anomalies = (
        libelle_attendu.isna()
        | libelle_attendu.ne(mos[f"Poste {annee}"]).fillna(True)
    )

    print(f"Anomalies code/libellé en {annee} :", anomalies.sum())

# Géométries absentes
geometries_absentes = mos["Geo Shape"].isna()

print("\nGéométries absentes :", geometries_absentes.sum())
print("Part des lignes concernées (%) :",
      round(geometries_absentes.mean() * 100, 2))

# Poids des géométries absentes dans la surface renseignée
surface_mos = pd.to_numeric(mos["st_area_sh"], errors="coerce")

part_surface = (
    surface_mos[geometries_absentes].sum()
    / surface_mos.sum()
    * 100
)

print("Part de la surface renseignée concernée (%) :",
      round(part_surface, 2))

print("Surfaces absentes, non numériques ou non positives :",
      (surface_mos.isna() | surface_mos.le(0)).sum())

# Vérifier les classes présentes dans l'extraction
print("\nClasses présentes en 2025 :")
display(mos["Poste 2025"].value_counts())

# Couverture par département
couverture_mos = (
    mos.assign(
        departement=mos["insee"].str[:2],
        geometrie_absente=geometries_absentes
    )
    .groupby("departement")
    .agg(
        zones=("insee", "size"),
        geometries_absentes=("geometrie_absente", "sum")
    )
)

display(couverture_mos)

Anomalies code/libellé en 2021 : 0
Anomalies code/libellé en 2025 : 0

Géométries absentes : 305
Part des lignes concernées (%) : 0.72
Part de la surface renseignée concernée (%) : 11.96
Surfaces absentes, non numériques ou non positives : 0

Classes présentes en 2025 :


,count
Poste 2025,
Bois ou forêts,42115


,zones,geometries_absentes
departement,,
75,164,0
77,20237,146
78,8962,70
91,7270,53
92,220,1
93,184,0
94,355,2
95,4723,33


In [17]:
# Correspondance entre codes MOS et libellés
dictionnaire_mos = nomenclature.set_index("ID")["Poste"]

for annee in ["2021", "2025"]:
    libelle_attendu = mos[f"mos{annee}"].map(dictionnaire_mos)

    anomalies = (
        libelle_attendu.isna()
        | libelle_attendu.ne(mos[f"Poste {annee}"]).fillna(True)
    )

    print(f"Anomalies code/libellé en {annee} :", anomalies.sum())

# Géométries absentes
geometries_absentes = mos["Geo Shape"].isna()

print("\nGéométries absentes :", geometries_absentes.sum())
print("Part des lignes concernées (%) :",
      round(geometries_absentes.mean() * 100, 2))

# Poids des géométries absentes dans la surface renseignée
surface_mos = pd.to_numeric(mos["st_area_sh"], errors="coerce")

part_surface = (
    surface_mos[geometries_absentes].sum()
    / surface_mos.sum()
    * 100
)

print("Part de la surface renseignée concernée (%) :",
      round(part_surface, 2))

print("Surfaces absentes, non numériques ou non positives :",
      (surface_mos.isna() | surface_mos.le(0)).sum())

import json

nombre_illisibles = 0

for geometrie in mos["Geo Shape"].dropna():
    try:
        json.loads(geometrie)
    except (ValueError, TypeError):
        nombre_illisibles += 1

print("Nombre de JSON illisibles :", nombre_illisibles)

# Vérifier les classes présentes dans l'extraction
print("\nClasses présentes en 2025 :")
display(mos["Poste 2025"].value_counts())

# Couverture par département
couverture_mos = (
    mos.assign(
        departement=mos["insee"].str[:2],
        geometrie_absente=geometries_absentes
    )
    .groupby("departement")
    .agg(
        zones=("insee", "size"),
        geometries_absentes=("geometrie_absente", "sum")
    )
)

display(couverture_mos)

Anomalies code/libellé en 2021 : 0
Anomalies code/libellé en 2025 : 0

Géométries absentes : 305
Part des lignes concernées (%) : 0.72
Part de la surface renseignée concernée (%) : 11.96
Surfaces absentes, non numériques ou non positives : 0
Nombre de JSON illisibles : 0

Classes présentes en 2025 :


,count
Poste 2025,
Bois ou forêts,42115


,zones,geometries_absentes
departement,,
75,164,0
77,20237,146
78,8962,70
91,7270,53
92,220,1
93,184,0
94,355,2
95,4723,33


3.2 Analyse l’artificialisation départementale

In [13]:
colonnes_numeriques = [
    "pourcent_artif_1", "pourcent_artif_2",
    "surface_artif_1", "surface_artif_2",
    "flux_surface_1_2", "flux_percent_1_2"
]

artif_num = artif[colonnes_numeriques].apply(
    pd.to_numeric, errors="coerce"
)

print("Valeurs numériques absentes ou invalides :",
      artif_num.isna().sum().sum())

print("Doublons de code département :",
      artif["departement_code"].duplicated().sum())

# Vérification des pourcentages
pourcentages_invalides = (
    ~artif_num["pourcent_artif_1"].between(0, 100)
    | ~artif_num["pourcent_artif_2"].between(0, 100)
)

print("Lignes avec pourcentage invalide :",
      pourcentages_invalides.sum())

# Cohérence des flux
ecart_surface = (
    artif_num["surface_artif_2"]
    - artif_num["surface_artif_1"]
    - artif_num["flux_surface_1_2"]
).abs()

ecart_points = (
    artif_num["pourcent_artif_2"]
    - artif_num["pourcent_artif_1"]
    - artif_num["flux_percent_1_2"]
).abs()

print("Flux de surface incohérents :", (ecart_surface > 0.000001).sum())
print("Flux en points incohérents :", (ecart_points > 0.0000000001).sum())

# Sélection de l'Île-de-France
artif_idf = artif[artif["region_code"] == "11"].copy()

artif_idf["Duree"] = (
    pd.to_numeric(artif_idf["millesimes_2"])
    - pd.to_numeric(artif_idf["millesimes_1"])
)

print("\nNombre de départements franciliens :", len(artif_idf))

display(artif_idf[[
    "departement_code", "nom",
    "millesimes_1", "millesimes_2",
    "Duree", "pourcent_artif_2"
]])

Valeurs numériques absentes ou invalides : 0
Doublons de code département : 0
Lignes avec pourcentage invalide : 0
Flux de surface incohérents : 0
Flux en points incohérents : 0

Nombre de départements franciliens : 8


,departement_code,nom,millesimes_1,millesimes_2,Duree,pourcent_artif_2
75,75,Paris,2018,2021,3,83.3099967600104
77,77,Seine-et-Marne,2017,2021,4,12.0362451633144
78,78,Yvelines,2018,2021,3,20.8031283350811
91,91,Essonne,2018,2021,3,22.8165381405166
92,92,Hauts-de-Seine,2018,2021,3,78.5474202155205
93,93,Seine-Saint-Denis,2018,2021,3,83.3030148100423
94,94,Val-de-Marne,2018,2021,3,74.8908062167845
95,95,Val-d'Oise,2018,2021,3,24.8555055285997


3.3 Analyse les trois fichiers météo

In [14]:
departements_idf = {"75", "77", "78", "91", "92", "93", "94", "95"}

archives_meteo = {
    2024: meteo_2024,
    2025: meteo_2025,
    2026: meteo_2026
}

distributions = []

for annee, base in archives_meteo.items():
    # Normalisation du code département
    base["num_dep"] = base["num_dep"].str.zfill(2)

    dates = pd.to_datetime(base["date"], utc=True, errors="coerce")

    base["jour_emission"] = (
        dates.dt.tz_convert("Europe/Paris")
        .dt.tz_localize(None)
        .dt.normalize()
    )

    print(f"\n--- Météo {annee} ---")
    print("Première date :", base["jour_emission"].min())
    print("Dernière date :", base["jour_emission"].max())
    print("Dates invalides :", dates.isna().sum())

    doublons = base.duplicated(["num_dep", "jour_emission"])
    print("Doublons département/jour :", doublons.sum())

    # Grille attendue : 8 départements × jours de la fenêtre
    calendrier = pd.date_range(
        base["jour_emission"].min(),
        base["jour_emission"].max()
    )

    attendu = pd.MultiIndex.from_product(
        [sorted(departements_idf), calendrier],
        names=["num_dep", "jour_emission"]
    )

    idf = base[base["num_dep"].isin(departements_idf)]

    observe = pd.MultiIndex.from_frame(
        idf[["num_dep", "jour_emission"]].drop_duplicates()
    )

    print("Couples département/jour IDF absents :",
          len(attendu.difference(observe)))

    for horizon in ["niveau_j1", "niveau_j2"]:
        niveaux = pd.to_numeric(base[horizon], errors="coerce")

        print(f"Valeurs invalides dans {horizon} :",
              (~niveaux.isin([1, 2, 3, 4])).sum())

        effectifs = pd.to_numeric(
            idf[horizon], errors="coerce"
        ).value_counts()

        distributions.append({
            "Année": annee,
            "Horizon": horizon,
            **{f"Code {code}": int(effectifs.get(code, 0))
               for code in [1, 2, 3, 4]},
            "Total IDF": len(idf)
        })

distribution_meteo = pd.DataFrame(distributions)
display(distribution_meteo)


--- Météo 2024 ---
Première date : 2024-06-03 00:00:00
Dernière date : 2024-09-30 00:00:00
Dates invalides : 0
Doublons département/jour : 0
Couples département/jour IDF absents : 0
Valeurs invalides dans niveau_j1 : 0
Valeurs invalides dans niveau_j2 : 0

--- Météo 2025 ---
Première date : 2025-06-03 00:00:00
Dernière date : 2025-09-30 00:00:00
Dates invalides : 0
Doublons département/jour : 0
Couples département/jour IDF absents : 0
Valeurs invalides dans niveau_j1 : 0
Valeurs invalides dans niveau_j2 : 0

--- Météo 2026 ---
Première date : 2026-05-28 00:00:00
Dernière date : 2026-10-05 00:00:00
Dates invalides : 0
Doublons département/jour : 0
Couples département/jour IDF absents : 0
Valeurs invalides dans niveau_j1 : 0
Valeurs invalides dans niveau_j2 : 0


,Année,Horizon,Code 1,Code 2,Code 3,Code 4,Total IDF
0,2024,niveau_j1,958,2,0,0,960
1,2024,niveau_j2,959,1,0,0,960
2,2025,niveau_j1,837,120,3,0,960
3,2025,niveau_j2,841,116,3,0,960
4,2026,niveau_j1,461,418,168,1,1048
5,2026,niveau_j2,501,419,128,0,1048


3.4 Analyse BDIFF

In [15]:
dates_feux = pd.to_datetime(
    bdiff["Date de première alerte"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

surface_totale = pd.to_numeric(
    bdiff["Surface parcourue (m2)"], errors="coerce"
)

surface_foret = pd.to_numeric(
    bdiff["Surface forêt (m2)"], errors="coerce"
)

print("Première alerte :", dates_feux.min())
print("Dernière alerte :", dates_feux.max())
print("Dates invalides :", dates_feux.isna().sum())

print("Doublons de la clé Année + Numéro :",
      bdiff.duplicated(["Année", "Numéro"]).sum())

print("Années différentes de l'année de première alerte :",
      pd.to_numeric(bdiff["Année"]).ne(dates_feux.dt.year).sum())

print("Surfaces totales absentes, invalides ou négatives :",
      (surface_totale.isna() | surface_totale.lt(0)).sum())

print("Surfaces forêt supérieures au total :",
      surface_foret.gt(surface_totale).sum())

print("\nPrécision des surfaces :")
display(bdiff["Précision des surfaces"].fillna("Non renseignée").value_counts())

print("\nStatut source :")
display(bdiff["Précision de la donnée"].fillna("Champ vide").value_counts())

# Couverture des huit départements franciliens
couverture_bdiff = (
    bdiff["Département"]
    .value_counts()
    .reindex(sorted(departements_idf), fill_value=0)
)

display(couverture_bdiff)

# Repérer les événements identiques sur tous les champs sauf le numéro
doublons_potentiels = bdiff.drop(
    columns="Numéro"
).duplicated(keep=False)

display(bdiff.loc[doublons_potentiels, [
    "Année", "Numéro", "Code INSEE",
    "Date de première alerte", "Surface parcourue (m2)"
]])

Première alerte : 2006-07-26 15:05:00
Dernière alerte : 2025-11-05 20:44:34
Dates invalides : 0
Doublons de la clé Année + Numéro : 0
Années différentes de l'année de première alerte : 0
Surfaces totales absentes, invalides ou négatives : 0
Surfaces forêt supérieures au total : 0

Précision des surfaces :


,count
Précision des surfaces,
Estimées,276
Mesurées,61
Non renseignée,10



Statut source :


,count
Précision de la donnée,
Champ vide,336
Données partielles ou non valides,11


,count
Département,
75,0
77,293
78,5
91,39
92,0
93,0
94,1
95,9


,Année,Numéro,Code INSEE,Date de première alerte,Surface parcourue (m2)
42,2015,524,77186,2015-06-11 11:30:00,300
43,2015,523,77186,2015-06-11 11:30:00,300


Risque de multiplication lors de la jointure

In [16]:
polygones_par_commune = mos["insee"].value_counts()

nombre_lignes_jointure = (
    bdiff["Code INSEE"]
    .map(polygones_par_commune)
    .fillna(0)
    .sum()
)

print("Nombre de feux :", len(bdiff))
print("Lignes après jointure directe avec les polygones :",
      int(nombre_lignes_jointure))

print("Feux sans commune correspondante dans le MOS :",
      (~bdiff["Code INSEE"].isin(mos["insee"])).sum())

Nombre de feux : 347
Lignes après jointure directe avec les polygones : 42640
Feux sans commune correspondante dans le MOS : 0
